In [ ]:
# Importando bibliotecas

import pandas as pd
import cuml
import os
import scipy as sci
import time as tm
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.signal import decimate
import json

# Importando classificadores da biblioteca cuml

from cuml.ensemble import RandomForestClassifier #RF
from cuml.neighbors import KNeighborsClassifier #KNN
from cuml.svm import SVC #SVM
from cuml.naive_bayes import GaussianNB #GNB

# Importando PCA

from cuml.decomposition import PCA

# Importando separador dos dados, preprocessadores e métricas
from sklearn.model_selection import GridSearchCV, KFold, train_test_split, cross_val_score, cross_validate
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.metrics import ConfusionMatrixDisplay, f1_score, accuracy_score, confusion_matrix

# Importando classificados XGBoost

import xgboost
from xgboost import XGBClassifier

# Importando Rede Neural e skorch

import torch
import torch.nn as nn

from skorch import NeuralNetClassifier
from skorch.callbacks import EarlyStopping
from skorch.dataset import ValidSplit
from skorch.dataset import Dataset
from skorch.helper import predefined_split

In [ ]:
path_healthy = 'Healthy' #Caminho para acessar os dados saudáveis
path_damaged = 'Damaged' #Caminho para acessar os dados danificados

In [ ]:
# Função para preparar dados saudáveis
def prepare_healthy(path):
    files = ['H1.mat'] #Lista com arquivo do primeiro minuto dos dados saudáveis
    list_df_healthy = [] #Cria lista vazia
    for i in files:
        file_path = os.path.join(path, i) #Criando o caminho com a pasta e o nome do arquivo
        H = sci.io.loadmat(file_path) #Abre o arquivo .mat
        H = {k: v for k, v in H.items() if not k.startswith('__')} #Seleciona as colunas com dados
        df = pd.DataFrame({k: v.squeeze() for k, v in H.items()}) #Transforma em dataframe pandas
        df = df.drop(columns=['Speed']) #Remove coluna Speed
        list_df_healthy.append(df) #Adiciona o dataframe na lista
        print(i)
    return list_df_healthy #Retorna lista com o dataframe

# Função para preparar dados danificados
def prepare_damaged(path):
    files = ['D1.mat'] #Lista com arquivo do primeiro minuto dos dados danificados
    list_df_damaged = [] #Cria lista vazia
    for i in files:
        file_path = os.path.join(path, i) #Criando o caminho com a pasta e o nome do arquivo
        D = sci.io.loadmat(file_path) #Abre o arquivo .mat
        D = {k: v for k, v in D.items() if not k.startswith('__')} #Seleciona as colunas com dados
        df = pd.DataFrame({k: v.squeeze() for k, v in D.items()}) #Transforma em dataframe pandas
        df = df.drop(columns=['Speed', 'Torque']) #Remove as colunas Speed e Torque
        list_df_damaged.append(df) #Adiciona o dataframe na lista
        print(i)
    return list_df_damaged #Retorna lista com o dataframe

In [ ]:
healthy = prepare_healthy(path_healthy) #Chama função e cria lista com dataframes dos dados saudáveis

H1.mat


In [ ]:
damaged = prepare_damaged(path_damaged) #Chama função e cria lista com dataframes dos dados danificados

D1.mat


In [ ]:
time = np.linspace(0,60,60*40000) #Cria espaço de tempo de 0 a 60, com 2400000 valores

In [ ]:
for i in healthy:
    i['Time'] = time #Adiciona a coluna tempo

In [ ]:
for i in damaged:
    i['Time'] = time #Adiciona a coluna tempo

In [ ]:
#Função para redução de frequência com decimate do scipy

def downsample(df, freq):
    df_num = df.select_dtypes(include=[np.number]).drop(columns=['Time']) #Seleciona apenas dados numéricos, removendo a coluna tempo
    step = 40000//freq #Calcula a razão entre a frequência original e a nova

    data = {} #Cria dicionaário vazio

    #A razão é verificada se é maior do que 13
    if step<=13:
        print('Redução em uma etapa')
        for col in df_num.columns:
            data[col] = decimate(df_num[col].values, q=step, axis=0, ftype='fir', zero_phase=True) #Realiza a redução para a nova frequência
    else:
        # Para a frequência de 1 kHz, a razão é maior do que 13, sendo assim realizada em duas etapas
        print('Redução em duas etapas: 8 e 5')
        for col in df_num.columns:
            data[col] = decimate(df_num[col].values, q=8, axis=0, ftype='fir', zero_phase=True) #Primeira redução
            data[col] = decimate(data[col], q=5, axis=0, ftype='fir', zero_phase=True) #Segunda redução

    df_down=pd.DataFrame(data) #Cria dataframe com os dados reduzidos
    df_down['time'] = df['Time'].iloc[::step].values #Adiciona os dados de tempo adequados para a redução de dados

    return df_down #Retorna dataframe com os dados reduzidos

In [ ]:
#Rede Neural

class neural_network_2(nn.Module):
  def __init__(self, in_features):
    super().__init__()
    self.flatten = nn.Flatten()
    self.linear_relu_stack = nn.Sequential(
        nn.Linear(in_features,int(in_features*2)), #Cria camada de entrada
        nn.ReLU(), #Função de ativação ReLU
        nn.Linear(int(in_features*2),8), #Segunda camada
        nn.ReLU(), #Função de ativação ReLU
        nn.Linear(8,4), #Terceira camada
        nn.ReLU(), #Função de ativação ReLU
        nn.Linear(4,1), #Camada de saída
        nn.Sigmoid()) # Função sigmoíde

  def forward(self, x):
    x = self.flatten(x)
    logits = self.linear_relu_stack(x)
    return logits.squeeze(-1)

In [ ]:
#Função para preparação os dados

def prepare_transform_X_y(healthy_df, damaged_df, freq, use_pca = False):
    downsample_healthy = downsample(healthy_df, freq) #Reduz os dados saudáveis de acordo com a frequência

    downsample_damaged = downsample(damaged_df, freq) #Reduz os dados danificados de acordo com a frequência

    downsample_healthy['Target'] = ['Healthy']*len(downsample_healthy) #Cria coluna de target com label saudável nos dados saudáveis

    downsample_damaged['Target'] = ['Damaged']*len(downsample_damaged) #Cria coluna de target com label danificado nos dados danificados

    df_join = pd.concat([downsample_healthy, downsample_damaged]) #Junta os dados danificados e saudáveis

    df_join = df_join.sort_values(by='time') #Ordena os valores pelo tempo

    df_test = df_join.drop(columns=['time']) #Remove a coluna de tempo

    X = df_test.iloc[:,:-1].values #Separa os dados classificadores
    y = df_test.iloc[:,-1].values #Separa os targets

    sc = StandardScaler() # Inicia normalizador
    X_norm = sc.fit_transform(X) #Normaliza os dados

    encoder = LabelEncoder() #Inicia encoder para os dados categóricos de target
    y_le = encoder.fit_transform(y) #Transforma dados categóricos em numéricos

    if use_pca == True: #Verifica se a variável use_pca é igual a True
        pca_i = PCA() # Inicia função PCA
        X_pca_i = pca_i.fit_transform(X_norm) #Transforma os dados normalizados
        comps = np.argmax(np.cumsum(pca_i.explained_variance_ratio_)>0.8)+1 #Calcula o número de componentes necessários para uma variância explicada de 80%
        pca = PCA(n_components=comps) #Inicia nova função PCA com o número de componentes
        X_pca = pca.fit_transform(X_norm) #Transforma os dados padronizados

        return [X_pca, y_le] #Retorna os dados classificadores com PCA e targets

    else:
        return [X_norm, y_le] #Retorna os dados normalizados e targets
    
# Função para validação cruzada

def cross_val_models_results(model, X, y, kf):
    cross_val = cross_validate(model, X, y, cv=kf, verbose=2) #Realiza a validação cruzada dos dados

    cross_val_dict = pd.DataFrame(cross_val).to_dict() #Transforma o resultado em dicionário

    return cross_val_dict #Retorna o dicionário com o resultado da validação cruzada

#Função para obter os resultados

def results_freq_dist(models_list, models_name_l, healthy_df, damaged_df, freq, splits, file_export_name, pca= False):
    if pca == True:
        print('Using PCA')
    else:
        print('Not using PCA')
    X, y = prepare_transform_X_y(healthy_df, damaged_df, freq, use_pca= pca) #Obtém os dados
    X = X.astype(np.float32) #Transforma os dados em float32
    y = y.astype(np.float32) #Transforma os dados em float32
    print(f'Data array shape: {X.shape}')
    print(np.prod(X.shape[1:]))
    kf = KFold(n_splits=splits, shuffle=True, random_state=42) #Inicializa kFold com embaralhamento dos dados, e estado aleatório 42
    results = {} #Cria dicionário vazio

    for i in range(len(models_list)):
        print(f'start: {models_name_l[i]}')
        if models_name_l[i]=='NN': #Checa se o modelo é rede neural
            models_list[i].set_params(module__in_features = np.prod(X.shape[1:])) #Define o número de features de entrada, como o número de colunas dos dados
        results_i = cross_val_models_results(models_list[i], X, y, kf=kf) #Obtém os resultados para o modelo
        results[models_name_l[i]] = results_i #Adiciona os resultados no dicionário
        print(f'end: {models_name_l[i]}')
        print('--------------------------------')

    with open(file_export_name, 'w') as f:
        json.dump(results, f, indent=4) #Exporta os resultados em formato .json

    return results #Retorna os resultados

#Função para definir os modelos e parâmetros

def get_results(healthy_df, damaged_df, freq, splits, file_export_name, pca= False):
    net_2 = NeuralNetClassifier(neural_network_2,
                            criterion=nn.BCELoss(), train_split=ValidSplit(0.2,random_state=42),
                            max_epochs=40, callbacks=[EarlyStopping(patience=10, threshold=0.01,load_best=True)],
                            lr=0.1, verbose=1) #Cria a rede neural com skorch, utilizando o critério de parada antecipada, com 20% de dados de validação e estado aleatório 42

    models_ml = [GaussianNB(), KNeighborsClassifier(n_neighbors = 9),
             RandomForestClassifier(criterion = 'gini', max_depth = 15, n_estimators= 150),
             SVC(C= 1, gamma= 0.1), XGBClassifier(learning_rate= 0.1, max_depth= 5, n_estimators= 200),
             net_2] #Lista com os modelos de aprendizado com os parâmetros definidos

    models_name = ['Gauss', 'KNN', 'RF', 'SVM', 'XGBoost', 'NN'] #Lista com os nomes dos modelos

    results = results_freq_dist(models_ml, models_name,healthy_df,
                               damaged_df, freq = freq, splits = splits, pca = pca,
                               file_export_name = file_export_name) #Calacula os resultados para os modelos

    return results #Retorna os resultados

In [ ]:
results_20 = get_results(healthy[0], damaged[0], freq = 20000,
                               splits = 5, pca = False,
                               file_export_name = 'results_dist_20_2.json') #Calcula os resultados para a frequência de 20 kHz, sem utilizar PCA, exportando os resultados

Not using PCA
Redução em uma etapa
Redução em uma etapa
Data array shape: (2400000, 8)
8
start: Gauss
[CV] END .................................................... total time=   0.4s
[CV] END .................................................... total time=   0.2s
[CV] END .................................................... total time=   0.2s
[CV] END .................................................... total time=   0.2s
[CV] END .................................................... total time=   0.2s
end: Gauss
--------------------------------
start: KNN


[Parallel(n_jobs=1)]: Done   5 out of   5 | elapsed:    1.3s finished


[CV] END .................................................... total time=  11.0s
[CV] END .................................................... total time=  11.1s
[CV] END .................................................... total time=  11.1s
[CV] END .................................................... total time=  11.2s
[CV] END .................................................... total time=  11.2s
end: KNN
--------------------------------
start: RF


[Parallel(n_jobs=1)]: Done   5 out of   5 | elapsed:   55.8s finished


[CV] END .................................................... total time=  11.0s
[CV] END .................................................... total time=  10.9s
[CV] END .................................................... total time=  11.0s
[CV] END .................................................... total time=  11.0s
[CV] END .................................................... total time=  10.9s
end: RF
--------------------------------
start: SVM


...

[Parallel(n_jobs=1)]: Done   5 out of   5 | elapsed: 13.2min finished


[CV] END .................................................... total time=   6.6s
[CV] END .................................................... total time=   6.5s
[CV] END .................................................... total time=   6.4s
[CV] END .................................................... total time=   6.7s
[CV] END .................................................... total time=   6.4s
end: XGBoost
--------------------------------
start: NN


[Parallel(n_jobs=1)]: Done   5 out of   5 | elapsed:   32.8s finished


  epoch    train_loss    valid_acc    valid_loss      dur
-------  ------------  -----------  ------------  -------
      1        0.1759       0.9424        0.1535  26.8092
      2        0.1529       0.9434        0.1513  26.9335
      3        0.1514       0.9436        0.1505  26.8627
      4        0.1507       0.9437        0.1501  26.8918
      5        0.1503       0.9439        0.1498  27.3103
      6        0.1500       0.9440        0.1497  27.0531
      7        0.1497       0.9439        0.1495  26.9772
      8        0.1495       0.9441        0.1493  27.0351
      9        0.1494       0.9441        0.1492  27.0860
     10        0.1492       0.9442        0.1491  27.0790
     11        0.1491       0.9442        0.1490  27.4923
     12        0.1490       0.9443        0.1489  27.3176
     13        0.1490       0.9442        0.1489  27.1453
     14        0.1489       0.9443        0.1488  26.9493
     15        0.1489       0.9444        0.1487  27.6137
Stopping since

[Parallel(n_jobs=1)]: Done   5 out of   5 | elapsed: 41.4min finished


In [ ]:
results_10 = get_results(healthy[0], damaged[0], freq = 10000,
                         splits = 5, pca = False,
                         file_export_name = 'results_dist_10_2.json') #Calcula os resultados para a frequência de 10 kHz, sem utilizar PCA, exportando os resultados

Not using PCA
Redução em uma etapa
Redução em uma etapa
Data array shape: (1200000, 8)
8
start: Gauss
[CV] END .................................................... total time=   0.1s
[CV] END .................................................... total time=   0.1s
[CV] END .................................................... total time=   0.1s
[CV] END .................................................... total time=   0.1s
[CV] END .................................................... total time=   0.1s
end: Gauss
--------------------------------
start: KNN


[Parallel(n_jobs=1)]: Done   5 out of   5 | elapsed:    0.5s finished


[CV] END .................................................... total time=   2.7s
[CV] END .................................................... total time=   2.7s
[CV] END .................................................... total time=   2.7s
[CV] END .................................................... total time=   2.8s
[CV] END .................................................... total time=   2.8s
end: KNN
--------------------------------
start: RF


[Parallel(n_jobs=1)]: Done   5 out of   5 | elapsed:   13.8s finished


[CV] END .................................................... total time=   4.8s
[CV] END .................................................... total time=   4.8s
[CV] END .................................................... total time=   4.8s
[CV] END .................................................... total time=   4.9s
[CV] END .................................................... total time=   4.8s
end: RF
--------------------------------
start: SVM


...

[Parallel(n_jobs=1)]: Done   5 out of   5 | elapsed:  3.6min finished


[CV] END .................................................... total time=   3.3s
[CV] END .................................................... total time=   3.3s
[CV] END .................................................... total time=   3.3s
[CV] END .................................................... total time=   3.3s
[CV] END .................................................... total time=   3.3s
end: XGBoost
--------------------------------
start: NN


[Parallel(n_jobs=1)]: Done   5 out of   5 | elapsed:   16.7s finished


  epoch    train_loss    valid_acc    valid_loss      dur
-------  ------------  -----------  ------------  -------
      1        0.1882       0.9331        0.1721  13.7396
      2        0.1645       0.9348        0.1693  13.5654
      3        0.1628       0.9362        0.1676  13.6474
      4        0.1620       0.9369        0.1665  13.6454
      5        0.1614       0.9371        0.1658  13.5838
      6        0.1609       0.9375        0.1653  13.7549
      7        0.1606       0.9376        0.1649  13.8136
      8        0.1603       0.9378        0.1641  13.5992
      9        0.1601       0.9380        0.1636  13.4790
     10        0.1599       0.9385        0.1630  13.5890
     11        0.1597       0.9388        0.1625  13.6488
     12        0.1595       0.9387        0.1625  13.6385
     13        0.1594       0.9388        0.1621  13.7008
     14        0.1592       0.9389        0.1619  13.5424
     15        0.1591       0.9388        0.1619  13.5189
     16       

[Parallel(n_jobs=1)]: Done   5 out of   5 | elapsed: 21.9min finished


In [ ]:
results_1 = get_results(healthy[0], damaged[0], freq = 1000,
                              splits = 5, pca = False,
                              file_export_name = 'results_dist_1_2.json') #Calcula os resultados para a frequência de 1 kHz, sem utilizar PCA, exportando os resultados

Not using PCA
Redução em duas etapas: 8 e 5
Redução em duas etapas: 8 e 5
Data array shape: (120000, 8)
8
start: Gauss
[CV] END .................................................... total time=   0.0s
[CV] END .................................................... total time=   0.0s
[CV] END .................................................... total time=   0.0s
[CV] END .................................................... total time=   0.0s
[CV] END .................................................... total time=   0.0s
end: Gauss
--------------------------------
start: KNN
[CV] END .................................................... total time=   0.0s
[CV] END .................................................... total time=   0.0s


[Parallel(n_jobs=1)]: Done   5 out of   5 | elapsed:    0.1s finished
[Parallel(n_jobs=1)]: Done   5 out of   5 | elapsed:    0.2s finished


[CV] END .................................................... total time=   0.0s
[CV] END .................................................... total time=   0.0s
[CV] END .................................................... total time=   0.0s
end: KNN
--------------------------------
start: RF
[CV] END .................................................... total time=   0.6s
[CV] END .................................................... total time=   0.6s
[CV] END .................................................... total time=   0.6s
[CV] END .................................................... total time=   0.6s
[CV] END .................................................... total time=   0.6s
end: RF
--------------------------------
start: SVM


[Parallel(n_jobs=1)]: Done   5 out of   5 | elapsed:    3.1s finished


[CV] END .................................................... total time=   0.2s
[CV] END .................................................... total time=   0.2s
[CV] END .................................................... total time=   0.2s
[CV] END .................................................... total time=   0.2s
[CV] END .................................................... total time=   0.2s
end: SVM
--------------------------------
start: XGBoost


[Parallel(n_jobs=1)]: Done   5 out of   5 | elapsed:    1.0s finished


[CV] END .................................................... total time=   0.5s
[CV] END .................................................... total time=   0.5s
[CV] END .................................................... total time=   0.5s
[CV] END .................................................... total time=   0.5s
[CV] END .................................................... total time=   0.5s
end: XGBoost
--------------------------------
start: NN


[Parallel(n_jobs=1)]: Done   5 out of   5 | elapsed:    2.3s finished


  epoch    train_loss    valid_acc    valid_loss     dur
-------  ------------  -----------  ------------  ------
      1        0.5648       0.7235        0.5119  1.3659
      2        0.4298       0.9732        0.1306  1.3732
      3        0.0761       0.9862        0.0505  1.3633
      4        0.0462       0.9884        0.0386  1.3720
      5        0.0386       0.9892        0.0336  1.3813
      6        0.0350       0.9898        0.0311  1.3769
      7        0.0331       0.9895        0.0299  1.3496
      8        0.0319       0.9899        0.0287  1.3531
      9        0.0310       0.9901        0.0283  1.3482
     10        0.0303       0.9902        0.0280  1.3675
     11        0.0297       0.9903        0.0277  1.3550
     12        0.0293       0.9902        0.0277  1.3446
     13        0.0290       0.9903        0.0277  1.4006
     14        0.0286       0.9904        0.0275  1.3667
     15        0.0284       0.9903        0.0273  1.3651
     16        0.0282       0.9

[Parallel(n_jobs=1)]: Done   5 out of   5 | elapsed:  3.9min finished


In [ ]:
results_20_pca = get_results(healthy[0], damaged[0], freq = 20000,
                         splits = 5, pca = True,
                         file_export_name = 'results_dist_20_pca_2.json') #Calcula os resultados para a frequência de 20 kHz, utilizando PCA, exportando os resultados

Using PCA
Redução em uma etapa
Redução em uma etapa
[2026-02-14 16:43:26.113] [CUML] [warning] Warning(`fit`): As of v0.16, PCA invoked without an n_components argument defaults to using min(n_samples, n_features) rather than 1
Data array shape: (2400000, 6)
6
start: Gauss
[CV] END .................................................... total time=   0.1s
[CV] END .................................................... total time=   0.1s
[CV] END .................................................... total time=   0.1s
[CV] END .................................................... total time=   0.1s
[CV] END .................................................... total time=   0.1s
end: Gauss
--------------------------------
start: KNN


[Parallel(n_jobs=1)]: Done   5 out of   5 | elapsed:    0.9s finished


[CV] END .................................................... total time=  10.9s
[CV] END .................................................... total time=  11.0s
[CV] END .................................................... total time=  11.0s
[CV] END .................................................... total time=  11.1s
[CV] END .................................................... total time=  11.1s
end: KNN
--------------------------------
start: RF


[Parallel(n_jobs=1)]: Done   5 out of   5 | elapsed:   55.4s finished


[CV] END .................................................... total time=  10.3s
[CV] END .................................................... total time=  10.4s
[CV] END .................................................... total time=  10.4s
[CV] END .................................................... total time=  10.3s
[CV] END .................................................... total time=  10.4s
end: RF
--------------------------------
start: SVM


...

[Parallel(n_jobs=1)]: Done   5 out of   5 | elapsed: 20.1min finished


[CV] END .................................................... total time=   6.4s
[CV] END .................................................... total time=   6.2s
[CV] END .................................................... total time=   6.2s
[CV] END .................................................... total time=   6.2s
[CV] END .................................................... total time=   6.2s
end: XGBoost
--------------------------------
start: NN


[Parallel(n_jobs=1)]: Done   5 out of   5 | elapsed:   31.3s finished


  epoch    train_loss    valid_acc    valid_loss      dur
-------  ------------  -----------  ------------  -------
      1        0.2459       0.9111        0.2238  26.8313
      2        0.2249       0.9120        0.2224  26.9843
      3        0.2236       0.9122        0.2219  26.8144
      4        0.2230       0.9123        0.2214  27.2185
      5        0.2227       0.9124        0.2212  26.8264
      6        0.2225       0.9124        0.2212  26.8148
      7        0.2223       0.9123        0.2212  27.0087
      8        0.2222       0.9124        0.2211  27.0353
      9        0.2221       0.9124        0.2211  27.1133
     10        0.2220       0.9125        0.2210  27.1807
     11        0.2219       0.9125        0.2210  27.0122
     12        0.2219       0.9125        0.2210  27.0076
     13        0.2218       0.9125        0.2209  27.0016
Stopping since valid_loss has not improved in the last 10 epochs.
Restoring best model from epoch 4.
[CV] END ....................

[Parallel(n_jobs=1)]: Done   5 out of   5 | elapsed: 33.9min finished


In [ ]:
results_10_pca = get_results(healthy[0], damaged[0], freq = 10000,
                         splits = 5, pca = True,
                         file_export_name = 'results_dist_10_pca_2.json') #Calcula os resultados para a frequência de 10 kHz, utilizando PCA, exportando os resultados

Using PCA
Redução em uma etapa
Redução em uma etapa
[2026-02-14 17:40:39.671] [CUML] [warning] Warning(`fit`): As of v0.16, PCA invoked without an n_components argument defaults to using min(n_samples, n_features) rather than 1
Data array shape: (1200000, 6)
6
start: Gauss
[CV] END .................................................... total time=   0.1s
[CV] END .................................................... total time=   0.1s
[CV] END .................................................... total time=   0.1s
[CV] END .................................................... total time=   0.1s
[CV] END .................................................... total time=   0.1s
end: Gauss
--------------------------------
start: KNN


[Parallel(n_jobs=1)]: Done   5 out of   5 | elapsed:    0.5s finished


[CV] END .................................................... total time=   2.7s
[CV] END .................................................... total time=   2.7s
[CV] END .................................................... total time=   2.7s
[CV] END .................................................... total time=   2.7s
[CV] END .................................................... total time=   2.7s
end: KNN
--------------------------------
start: RF


[Parallel(n_jobs=1)]: Done   5 out of   5 | elapsed:   13.7s finished


[CV] END .................................................... total time=   4.8s
[CV] END .................................................... total time=   4.8s
[CV] END .................................................... total time=   4.8s
[CV] END .................................................... total time=   4.9s
[CV] END .................................................... total time=   4.9s
end: RF
--------------------------------
start: SVM


...

[Parallel(n_jobs=1)]: Done   5 out of   5 | elapsed:  5.4min finished


[CV] END .................................................... total time=   3.1s
[CV] END .................................................... total time=   3.0s
[CV] END .................................................... total time=   3.0s
[CV] END .................................................... total time=   3.0s
[CV] END .................................................... total time=   3.0s
end: XGBoost
--------------------------------
start: NN


[Parallel(n_jobs=1)]: Done   5 out of   5 | elapsed:   15.1s finished


  epoch    train_loss    valid_acc    valid_loss      dur
-------  ------------  -----------  ------------  -------
      1        0.2568       0.9029        0.2376  13.5360
      2        0.2284       0.9033        0.2371  13.4464
      3        0.2267       0.9050        0.2338  13.6471
      4        0.2257       0.9064        0.2311  13.4329
      5        0.2251       0.9072        0.2299  13.3423
      6        0.2247       0.9078        0.2290  13.4777
      7        0.2243       0.9084        0.2283  13.3376
      8        0.2240       0.9088        0.2279  13.4239
      9        0.2237       0.9093        0.2271  13.5455
     10        0.2235       0.9091        0.2277  13.5078
     11        0.2232       0.9095        0.2272  13.4644
     12        0.2230       0.9096        0.2271  13.7407
     13        0.2229       0.9099        0.2264  13.5051
     14        0.2228       0.9098        0.2262  13.5159
     15        0.2227       0.9097        0.2263  13.4644
     16       

[Parallel(n_jobs=1)]: Done   5 out of   5 | elapsed: 24.6min finished


In [ ]:
results_1_pca = get_results(healthy[0], damaged[0], freq = 1000,
                              splits = 5, pca = True,
                              file_export_name = 'results_dist_1_pca_2.json') #Calcula os resultados para a frequência de 1 kHz, utilizando PCA, exportando os resultados

Using PCA
Redução em duas etapas: 8 e 5
Redução em duas etapas: 8 e 5
[2026-02-14 18:12:47.889] [CUML] [warning] Warning(`fit`): As of v0.16, PCA invoked without an n_components argument defaults to using min(n_samples, n_features) rather than 1
Data array shape: (120000, 4)
4
start: Gauss
[CV] END .................................................... total time=   0.0s
[CV] END .................................................... total time=   0.0s
[CV] END .................................................... total time=   0.0s
[CV] END .................................................... total time=   0.0s
[CV] END .................................................... total time=   0.0s
end: Gauss
--------------------------------
start: KNN
[CV] END .................................................... total time=   0.0s
[CV] END .................................................... total time=   0.0s


[Parallel(n_jobs=1)]: Done   5 out of   5 | elapsed:    0.1s finished
[Parallel(n_jobs=1)]: Done   5 out of   5 | elapsed:    0.2s finished


[CV] END .................................................... total time=   0.0s
[CV] END .................................................... total time=   0.0s
[CV] END .................................................... total time=   0.0s
end: KNN
--------------------------------
start: RF
[CV] END .................................................... total time=   1.1s
[CV] END .................................................... total time=   1.1s
[CV] END .................................................... total time=   1.1s
[CV] END .................................................... total time=   1.1s
[CV] END .................................................... total time=   1.1s
end: RF
--------------------------------
start: SVM


[Parallel(n_jobs=1)]: Done   5 out of   5 | elapsed:    5.7s finished


[CV] END .................................................... total time=   0.8s
[CV] END .................................................... total time=   0.8s
[CV] END .................................................... total time=   0.9s
[CV] END .................................................... total time=   0.8s
[CV] END .................................................... total time=   0.8s
end: SVM
--------------------------------
start: XGBoost


[Parallel(n_jobs=1)]: Done   5 out of   5 | elapsed:    4.1s finished


[CV] END .................................................... total time=   0.4s
[CV] END .................................................... total time=   0.4s
[CV] END .................................................... total time=   0.4s
[CV] END .................................................... total time=   0.4s
[CV] END .................................................... total time=   0.4s
end: XGBoost
--------------------------------
start: NN


[Parallel(n_jobs=1)]: Done   5 out of   5 | elapsed:    1.8s finished


  epoch    train_loss    valid_acc    valid_loss     dur
-------  ------------  -----------  ------------  ------
      1        0.4993       0.8802        0.2844  1.3572
      2        0.2774       0.8841        0.2746  1.3321
      3        0.2722       0.8852        0.2705  1.3419
      4        0.2689       0.8870        0.2674  1.3403
      5        0.2667       0.8881        0.2657  1.3612
      6        0.2650       0.8894        0.2642  1.3412
      7        0.2638       0.8883        0.2635  1.3463
      8        0.2626       0.8891        0.2628  1.3354
      9        0.2616       0.8890        0.2622  1.3682
     10        0.2607       0.8892        0.2620  1.3434
     11        0.2599       0.8892        0.2614  1.3358
     12        0.2594       0.8899        0.2611  1.3703
     13        0.2591       0.8898        0.2611  1.3295
     14        0.2588       0.8899        0.2612  1.3379
     15        0.2587       0.8897        0.2608  1.3385
     16        0.2585       0.8

[Parallel(n_jobs=1)]: Done   5 out of   5 | elapsed:  2.5min finished
